# ARCHS4 CLAMPfull with C2 Canonical Pathways prior

💡 **Environment:** `clamp-analyses`  

## Load libraries

In [ ]:
if (!requireNamespace("CLAMP", quietly = TRUE)) {
    REPO_PATH <- "/home/msubirana/Documents/pivlab/CLAMP" 
    remotes::install_local(REPO_PATH, force = TRUE, dependencies = FALSE)
}

library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)

source(here("config.R"))

set.seed(config$ARCHS4$RANDOM_SVD_SEED)

## Output directory

In [ ]:
output_dir <- config$ARCHS4$DATASET_FOLDER
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

data_path <- here::here('data/archs4')
dir.create(data_path, showWarnings = FALSE, recursive = TRUE)

In [ ]:
# meta
meta <- readRDS(file.path(output_dir, "metadata_filtered.rds"))
n_genes_thin <- meta$n_genes_thin
n_samples <- meta$n_samples

archs4_genes <- meta$gene_symbols_thin

all_samples <- readRDS(file.path(output_dir, "all_samples.rds"))
sample_names <- all_samples[seq_len(n_samples)]

# fbm
fbm_file  <- file.path(output_dir, "fbm")
output_file <- paste0(fbm_file, "_filtered")

archs4_fbm_filt <- FBM(
  nrow        = n_genes_thin,
  ncol        = n_samples,
  backingfile = output_file,
  create_bk   = FALSE,
)

# svd
archs4_svdRes <- readRDS(file.path(output_dir, "svd.rds"))

In [ ]:
archs4_baseRes <- readRDS(file.path(output_dir, "archs4_baseRes.rds"))

In [ ]:
CLAMP_K_archs4 <- readRDS(file.path(output_dir, "CLAMP_K_archs4.rds"))

## Prepare C2 Canonical Pathways prior

In [ ]:
c2_gmt <- CLAMP:::read_gmt(file.path(data_path, "c2.cp.v2026.1.Hs.symbols.gmt"))
names(c2_gmt) <- paste0("C2CP_", names(c2_gmt))

c2_pathMat <- gmtListToSparseMat(list(C2CP = c2_gmt))

## CLAMPfull C2CP

In [ ]:
c2_matched <- getMatchedPathwayMat(c2_pathMat, archs4_genes)

archs4_fullRes <- CLAMPfull(
    Y = archs4_fbm_filt,
    svdres = archs4_svdRes,
    priorMat = c2_matched,
    clamp.base.result = archs4_baseRes,
    use_cpp = TRUE,
    trace = TRUE,
    clamp_k = CLAMP_K_archs4
  )

In [ ]:
archs4_fullRes$Z <- data.frame(archs4_baseRes$Z)
rownames(archs4_baseRes$Z) <- archs4_genes

archs4_fullRes$B <- data.frame(archs4_fullRes$B)
colnames(archs4_fullRes$B) <- sample_names

archs4_fullRes$summary <- archs4_fullRes$summary %>%
    dplyr::rename(LV = LV_index) %>%
    dplyr::mutate(LV = paste0('LV', LV))

saveRDS(archs4_fullRes, file = file.path(output_dir, "archs4_CLAMP_C2CP.rds"))

model_dir <- file.path(output_dir, "archs4_CLAMP_C2CP")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

write.csv(archs4_fullRes$B, file.path(model_dir, "B.csv"))
write.csv(archs4_fullRes$Z, file.path(model_dir, "Z.csv"))
write.csv(archs4_fullRes$summary, file.path(model_dir, "summary.csv"))